In [ ]:
import pandas as pd
import os
import gc
import time
from pathlib import Path
from src.common.minio_client import download_df_parquet
import wandb

WANDB_PROJECT  = "pd1-c2526-team5"
WANDB_RUN_NAME = "dcrnn-propagacion-estacion"



In [ ]:
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

## Dataset final

In [ ]:
df_final = download_df_parquet(access_key, secret_key,"grupo5/final/year=2025/month=01/dataset_final.parquet")


In [ ]:
df_final.stop_id.value_counts()

In [ ]:
(df_final[df_final.is_unscheduled == False].stop_id.value_counts() > 0).sum()

In [ ]:
df_final['stop_id'].str[:3].value_counts()

In [ ]:
df_final.columns

## GTFS Scheduled

In [ ]:
START_DATE = "2025-01-06"
END_DATE   = "2025-01-19"  # 2 semanas bastan para capturar toda la topología del grafo

dates = pd.date_range(start=START_DATE, end=END_DATE).strftime("%Y-%m-%d").tolist()
dfs = []
for date in dates:
    try:
        df_gtfs = download_df_parquet(
            access_key, secret_key,
            f"grupo5/cleaned/gtfs_clean_scheduled/date={date}/gtfs_scheduled_{date}.parquet"
        )
        dfs.append(df_gtfs)
    except Exception:
        print(f"No disponible: {date}")
        continue

if not dfs:
    raise RuntimeError("No se pudo descargar ningún archivo GTFS. Verifica las credenciales y la ruta.")

df = pd.concat(dfs, ignore_index=True)
del dfs  # liberar lista de DataFrames intermedios
print(f"GTFS cargado: {len(df):,} filas, {df['trip_uid'].nunique():,} viajes únicos")

## Cálculo de Matrices de adyacencia

In [ ]:
import numpy as np
import gc

# 1. Ordenar cronológicamente por viaje
df = df.sort_values(by=['trip_uid', 'scheduled_seconds']).reset_index(drop=True)

# 2. Calcular la parada siguiente dentro de cada viaje
df['next_stop_id']           = df.groupby('trip_uid')['stop_id'].shift(-1)
df['next_scheduled_seconds'] = df.groupby('trip_uid')['scheduled_seconds'].shift(-1)

# 3. Construir edges_df y liberar df inmediatamente para recuperar memoria
edges_df = df.dropna(subset=['next_stop_id']).copy()
del df
gc.collect()

# 4. Tiempo de viaje entre paradas consecutivas
edges_df['travel_time'] = edges_df['next_scheduled_seconds'] - edges_df['scheduled_seconds']

In [ ]:
# Eliminar tiempos de viaje inválidos
edges_df = edges_df[edges_df['travel_time'] > 0]

# Agregar aristas únicas con su tiempo mediano de viaje
graph_df = edges_df.groupby(['stop_id', 'next_stop_id']).agg(
    median_travel_time=('travel_time', 'median'),
    trip_count=('trip_uid', 'count')
).reset_index()

# edges_df ya no se necesita
del edges_df
gc.collect()

print(f"Aristas en el grafo: {len(graph_df):,}")

In [ ]:
# Nodos únicos: unión de orígenes y destinos de todas las aristas
nodes = sorted(set(graph_df['stop_id'].unique()) | set(graph_df['next_stop_id'].unique()))
n_nodes = len(nodes)

node_to_idx = {stop_id: idx for idx, stop_id in enumerate(nodes)}

# Vectorised fill — evita el loop lento de iterrows
src_idx = graph_df['stop_id'].map(node_to_idx).values
dst_idx = graph_df['next_stop_id'].map(node_to_idx).values

A_unweighted = np.zeros((n_nodes, n_nodes), dtype=np.float32)
A_unweighted[src_idx, dst_idx] = 1.0

print("Unweighted Adjacency Matrix Shape:", A_unweighted.shape)

In [ ]:
# Matriz de adyacencia ponderada (Gaussian Kernel sobre tiempo de viaje)
distances = graph_df['median_travel_time'].values
sigma = distances.std()

# Vectorised fill con guarda epsilon en denominador (evita sigma≈0 si todos los tiempos son iguales)
weights = np.exp(-(distances ** 2) / (sigma ** 2 + 1e-6))

A_weighted = np.zeros((n_nodes, n_nodes), dtype=np.float32)
A_weighted[src_idx, dst_idx] = weights

del graph_df
gc.collect()

print("Weighted Adjacency Matrix Shape:", A_weighted.shape)

In [ ]:
A_unweighted.sum()

In [ ]:
# scheduled_nodes_list ya está disponible como `nodes` del paso anterior
scheduled_nodes_list = nodes
print(f"Nodos en el grafo: {len(scheduled_nodes_list)}")

## Tensor generation

In [ ]:
import numpy as np

def build_spatiotemporal_tensor(df: pd.DataFrame, scheduled_nodes: list[str], freq: str = '15min'):
    """
    Transforms a flat event DataFrame into a (Time, Nodes, Features) tensor.
    Features: delay_seconds, lagged_delay_1, lagged_delay_2, is_unscheduled,
              temp_extreme, n_eventos_afectando, route_rolling_delay,
              actual_headway_seconds, hour_sin, hour_cos, dow,
              afecta_previo, afecta_durante, afecta_despues  (14 total)
    Targets:  station_delay_10m, station_delay_20m, station_delay_30m
    """
    # Early return si no hay nodos programados
    if not scheduled_nodes:
        raise ValueError("scheduled_nodes está vacío — no se puede construir el tensor.")

    # 1. Filtrar primero (subconjunto pequeño) y solo entonces copiar
    df = df[df['stop_id'].isin(scheduled_nodes)].copy()
    df['time_bin'] = pd.to_datetime(df['merge_time']).dt.floor(freq)

    # 2. Agregar eventos por ventana temporal y estación
    agg_rules = {
        'delay_seconds':          'mean',
        'lagged_delay_1':         'mean',
        'lagged_delay_2':         'mean',
        'is_unscheduled':         'sum',
        'temp_extreme':           'max',
        'n_eventos_afectando':    'max',
        'route_rolling_delay':    'mean',
        'actual_headway_seconds': 'mean',
        'afecta_previo':          'max',
        'afecta_durante':         'max',
        'afecta_despues':         'max',
        'station_delay_10m':      'mean', # se puede cambiar la variable objetivo
        'station_delay_20m':      'mean',
        'station_delay_30m':      'mean',
    }
    grouped_df = df.groupby(['time_bin', 'stop_id']).agg(agg_rules)
    del df
    gc.collect()

    # 3. Grid completo (producto cartesiano tiempo × nodos)
    all_times = pd.date_range(
        start=grouped_df.index.get_level_values('time_bin').min(),
        end=grouped_df.index.get_level_values('time_bin').max(),
        freq=freq
    )
    all_nodes = sorted(scheduled_nodes)
    full_index = pd.MultiIndex.from_product([all_times, all_nodes], names=['time_bin', 'stop_id'])
    full_df = grouped_df.reindex(full_index).reset_index()
    del grouped_df
    gc.collect()

    # 4. Imputación
    zero_fill_cols = ['delay_seconds', 'lagged_delay_1', 'lagged_delay_2',
                      'is_unscheduled', 'route_rolling_delay', 'actual_headway_seconds']
    full_df[zero_fill_cols] = full_df[zero_fill_cols].fillna(0)

    context_cols = ['temp_extreme', 'n_eventos_afectando', 'afecta_previo', 'afecta_durante', 'afecta_despues']
    full_df[context_cols] = full_df.groupby('stop_id')[context_cols].ffill()
    full_df[context_cols] = full_df.groupby('stop_id')[context_cols].bfill()
    full_df[context_cols] = full_df[context_cols].fillna(0)

    # 5. Features temporales derivadas del time_bin (sin ruido de agregación)
    full_df['hour_sin'] = np.sin(2 * np.pi * full_df['time_bin'].dt.hour / 24)
    full_df['hour_cos'] = np.cos(2 * np.pi * full_df['time_bin'].dt.hour / 24)
    full_df['dow']      = full_df['time_bin'].dt.dayofweek.astype(float)

    full_df = full_df.sort_values(['time_bin', 'stop_id'])

    feature_cols = [
        'delay_seconds', 'lagged_delay_1', 'lagged_delay_2',
        'is_unscheduled', 'temp_extreme', 'n_eventos_afectando',
        'route_rolling_delay', 'actual_headway_seconds',
        'hour_sin', 'hour_cos', 'dow',
        'afecta_previo', 'afecta_durante', 'afecta_despues',
    ]
    target_cols = ['station_delay_10m', 'station_delay_20m', 'station_delay_30m']

    T = len(all_times)
    N = len(all_nodes)

    # 6. Extraer arrays y liberar full_df antes de devolver
    X_tensor = full_df[feature_cols].values.reshape(T, N, len(feature_cols))
    Y_tensor = full_df[target_cols].values.reshape(T, N, len(target_cols))
    del full_df
    gc.collect()

    return X_tensor, Y_tensor, all_times, all_nodes


In [ ]:
import gc

X, Y, times, nodes = build_spatiotemporal_tensor(df_final, scheduled_nodes_list)
print("X Shape (Time, Nodes, Features):", X.shape)
print("Y Shape (Time, Nodes, Targets):", Y.shape)

# df_final ya no se necesita
del df_final
gc.collect()

## DCRNN Training

In [ ]:
# División cronológica (Train / Val / Test)
import numpy as np
import gc

X_np = np.nan_to_num(np.asarray(X), nan=0.0)
Y_np = np.nan_to_num(np.asarray(Y), nan=0.0)
del X, Y
gc.collect()

T = X_np.shape[0]
train_end = int(T * 0.70)
val_end   = train_end + int(T * 0.10)

X_train, X_val, X_test = X_np[:train_end], X_np[train_end:val_end], X_np[val_end:]
Y_train, Y_val, Y_test = Y_np[:train_end], Y_np[train_end:val_end], Y_np[val_end:]
del X_np, Y_np
gc.collect()

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Y_train: {Y_train.shape} | Y_val: {Y_val.shape} | Y_test: {Y_test.shape}")

In [ ]:
# Normalización de características y target (Feature & Target Scaling)
from sklearn.preprocessing import StandardScaler
import gc

# --- Escalar X ---
T_train, N, F = X_train.shape
X_train_2d = X_train.reshape(-1, F)
X_val_2d   = X_val.reshape(-1, F)
X_test_2d  = X_test.reshape(-1, F)

scaler_X = StandardScaler()
scaler_X.fit(X_train_2d)

X_train_scaled = scaler_X.transform(X_train_2d).reshape(X_train.shape)
X_val_scaled   = scaler_X.transform(X_val_2d).reshape(X_val.shape)
X_test_scaled  = scaler_X.transform(X_test_2d).reshape(X_test.shape)
del X_train, X_val, X_test, X_train_2d, X_val_2d, X_test_2d

# --- Escalar Y por separado ---
H_out = Y_train.shape[-1]
Y_train_2d = Y_train.reshape(-1, H_out)
Y_val_2d   = Y_val.reshape(-1, H_out)
Y_test_2d  = Y_test.reshape(-1, H_out)

scaler_Y = StandardScaler()
scaler_Y.fit(Y_train_2d)

Y_train_scaled = scaler_Y.transform(Y_train_2d).reshape(Y_train.shape)
Y_val_scaled   = scaler_Y.transform(Y_val_2d).reshape(Y_val.shape)
Y_test_scaled  = scaler_Y.transform(Y_test_2d).reshape(Y_test.shape)
del Y_train, Y_val, Y_test, Y_train_2d, Y_val_2d, Y_test_2d
gc.collect()

print("Escalado completado")
print(f"X_train_scaled: {X_train_scaled.shape}  (features: {F})")
print(f"Y_train_scaled: {Y_train_scaled.shape}  (targets: {H_out})")

In [ ]:
# Creacion de ventana deslizante (PyTorch Dataset)
import torch
from torch.utils.data import Dataset

class SubwayDataset(Dataset):
    def __init__(self, X, Y, history_len=4):
        self.X = torch.as_tensor(X.copy(), dtype=torch.float32)
        self.Y = torch.as_tensor(Y.copy(), dtype=torch.float32)
        self.history_len = history_len

        if self.X.shape[0] != self.Y.shape[0]:
            raise ValueError("X e Y deben tener el mismo tamano temporal (eje T).")
        if self.history_len < 1:
            raise ValueError("history_len debe ser >= 1.")
        if self.X.shape[0] <= self.history_len:
            raise ValueError("T debe ser mayor que history_len para generar ventanas.")

    def __len__(self):
        # Cada muestra usa [t, ..., t+history_len-1] para predecir t+history_len
        return self.X.shape[0] - self.history_len

    def __getitem__(self, idx):
        x_window = self.X[idx : idx + self.history_len]          # (history_len, N, F)
        y_window = self.Y[idx + self.history_len].unsqueeze(0)   # (1, N, H)
        return x_window, y_window

In [ ]:
# Generación de DataLoaders
from torch.utils.data import DataLoader

history_len = 8   # 2 horas de contexto a bins de 15min
batch_size  = 16  # Reducido a 16 para GPUs con 4 GB de VRAM

train_dataset = SubwayDataset(X_train_scaled, Y_train_scaled, history_len=history_len)
val_dataset   = SubwayDataset(X_val_scaled,   Y_val_scaled,   history_len=history_len)
test_dataset  = SubwayDataset(X_test_scaled,  Y_test_scaled,  history_len=history_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)   # shuffle=True para variabilidad de gradientes
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

x_batch, y_batch = next(iter(train_loader))
print(f"x_batch shape: {x_batch.shape}  # (Batch, History, Nodes, Features)")
print(f"y_batch shape: {y_batch.shape}  # (Batch, 1, Nodes, Horizontes)")


In [ ]:
# Preparación del Grafo (PyTorch Geometric Format)
import torch

print('Cuda available:', torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
A_tensor = torch.as_tensor(A_weighted, dtype=torch.float32)

# Matriz densa → representación COO: edge_index [2, E], edge_weight [E]
edge_index = torch.nonzero(A_tensor != 0, as_tuple=False).t().contiguous()
edge_weight = A_tensor[edge_index[0], edge_index[1]].contiguous()
del A_tensor

from torch_geometric.utils import add_self_loops, coalesce

# Auto-conexiones con peso 1.0 para evitar nans en la convolución
edge_index, edge_weight = add_self_loops(
    edge_index,
    edge_weight,
    fill_value=1.0,
    num_nodes=n_nodes
)

# Eliminar aristas duplicadas: DConv convierte internamente a denso y vuelve a sparse
# (dense_to_sparse), lo que deduplica. Si edge_index tiene duplicados, norm (calculado
# sobre E aristas) no coincide con reverse_edge_index (E-dups aristas) → RuntimeError.
edge_index, edge_weight = coalesce(edge_index, edge_weight, num_nodes=n_nodes)

edge_index  = edge_index.to(device)
edge_weight = edge_weight.to(device)

print(f"Device: {device}")
print(f"edge_index shape: {edge_index.shape}")
print(f"edge_weight shape: {edge_weight.shape}")
print(f"Número de aristas (sin duplicados, con self-loops): {edge_weight.numel()}")

In [ ]:
# Diagnóstico de alineación grafo ↔ datos
print("=== Diagnóstico de consistencia del grafo ===")
print(f"  n_nodes                   : {n_nodes}")
print(f"  X_train_scaled.shape[1]   : {X_train_scaled.shape[1]}")
print(f"  edge_index.max() + 1      : {edge_index.max().item() + 1}")
print(f"  edge_index.shape          : {tuple(edge_index.shape)}")
print(f"  edge_weight.shape         : {tuple(edge_weight.shape)}")
print(f"  edge_index.shape[1]       : {edge_index.shape[1]}")
print(f"  edge_weight.numel()       : {edge_weight.numel()}")
assert edge_index.shape[1] == edge_weight.numel(), "MISMATCH edge_index / edge_weight"
assert edge_index.max().item() + 1 == n_nodes,     "MISMATCH edge_index.max / n_nodes"
print("OK — sin duplicados ni desalineaciones.")
assert X_train_scaled.shape[1] == n_nodes, "MISMATCH datos (N) / grafo (n_nodes)"

In [ ]:
# Definición de la Arquitectura del Modelo
import torch.nn as nn
from torch_geometric_temporal.nn.recurrent import BatchedDCRNN

class SubwayDCRNN(nn.Module):
    def __init__(self, in_channels=14, hidden_channels=64, out_horizons=3, K=2, dropout=0.1):
        super().__init__()
        # BatchedDCRNN maneja el batch nativo y usa row/col swap para el grafo inverso,
        # evitando el bug de to_dense_adj/dense_to_sparse de DCRNN que requería grafo simétrico.
        self.dcrnn = BatchedDCRNN(in_channels=in_channels, out_channels=hidden_channels, K=K)
        self.readout = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, out_horizons)
        )

    def forward(self, X, edge_index, edge_weight):
        # X: (Batch, History, Nodes, Features)
        # BatchedDCRNN devuelve (Batch, History, Nodes, hidden_channels)
        out = self.dcrnn(X, edge_index, edge_weight)  # (B, H, N, hidden)
        hidden_state = out[:, -1, :, :]               # último paso temporal (B, N, hidden)
        y_hat = self.readout(hidden_state)             # (B, N, out_horizons)
        y_hat = y_hat.unsqueeze(1)                     # (B, 1, N, out_horizons)
        return y_hat


In [ ]:
# Instanciación y Configuración
IN_CHANNELS     = 14
HIDDEN_CHANNELS = 64
OUT_HORIZONS    = 3
K               = 2
DROPOUT         = 0.1
LR              = 1e-3
NUM_EPOCHS      = 50
ES_PATIENCE     = 7
GRAD_CLIP       = 3.0

model = SubwayDCRNN(
    in_channels=IN_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    out_horizons=OUT_HORIZONS,
    K=K,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, min_lr=1e-5
)
criterion = torch.nn.L1Loss()  # MAE en espacio escalado

wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        # Arquitectura
        "in_channels":     IN_CHANNELS,
        "hidden_channels": HIDDEN_CHANNELS,
        "out_horizons":    OUT_HORIZONS,
        "K":               K,
        "dropout":         DROPOUT,
        "history_len":     history_len,
        # Entrenamiento
        "lr":                      LR,
        "batch_size":              batch_size,
        "epochs":                  NUM_EPOCHS,
        "early_stopping_patience": ES_PATIENCE,
        "grad_clip":               GRAD_CLIP,
        # Datos
        "n_nodes":       n_nodes,
        "n_edges":       edge_index.shape[1],
        "train_samples": len(train_dataset),
        "val_samples":   len(val_dataset),
        "test_samples":  len(test_dataset),
        "targets":       ["station_delay_10m", "station_delay_20m", "station_delay_30m"],
        "features":      [
            "delay_seconds", "lagged_delay_1", "lagged_delay_2",
            "is_unscheduled", "temp_extreme", "n_eventos_afectando",
            "route_rolling_delay", "actual_headway_seconds",
            "hour_sin", "hour_cos", "dow",
            "afecta_previo", "afecta_durante", "afecta_despues",
        ],
    }
)

print(model)
print(f"\nDevice: {device}")
print(f"Optimizador: Adam | lr={LR}")
print(f"Scheduler: ReduceLROnPlateau (patience=3, factor=0.5)")
print(f"Loss: L1Loss (MAE) | Targets: station_delay_10m / 20m / 30m")

In [ ]:
print("=== Shape and Sanity Checks ===")
print(f"\nX tensors:")
print(f"  X_train_scaled: {X_train_scaled.shape}")
print(f"  X_val_scaled:   {X_val_scaled.shape}")
print(f"  X_test_scaled:  {X_test_scaled.shape}")

print(f"\nY tensors (scaled):")
print(f"  Y_train_scaled: {Y_train_scaled.shape}")
print(f"  Y_val_scaled:   {Y_val_scaled.shape}")
print(f"  Y_test_scaled:  {Y_test_scaled.shape}")

print(f"\nDataset windows (con history_len={history_len}):")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples:   {len(val_dataset)}")
print(f"  Test samples:  {len(test_dataset)}")

print(f"\nNaN checks:")
print(f"  X_train has NaNs: {np.isnan(X_train_scaled).any()}")
print(f"  Y_train has NaNs: {np.isnan(Y_train_scaled).any()}")
print(f"  X_test has NaNs:  {np.isnan(X_test_scaled).any()}")
print(f"  Y_test has NaNs:  {np.isnan(Y_test_scaled).any()}")

x_sample, y_sample = next(iter(train_loader))
print(f"\nBatch dimensions:")
print(f"  x_batch: {tuple(x_sample.shape)}  # (Batch, History, Nodes, Features)")
print(f"  y_batch: {tuple(y_sample.shape)}  # (Batch, 1, Nodes, Horizontes)")

In [ ]:
def validar_batch_vs_grafo(xb, yb, edge_index, edge_weight, tag=""):
    if edge_index.shape[1] != edge_weight.numel():
        raise RuntimeError(
            f"[{tag}] edge_index.shape[1]={edge_index.shape[1]} != edge_weight.numel()={edge_weight.numel()}"
        )
    n_graph = edge_index.max().item() + 1
    if xb.shape[2] != n_graph:
        raise RuntimeError(
            f"[{tag}] xb.shape[2]={xb.shape[2]} != n_graph={n_graph}"
        )
    if yb.shape[2] != n_graph:
        raise RuntimeError(
            f"[{tag}] yb.shape[2]={yb.shape[2]} != n_graph={n_graph}"
        )

In [ ]:
# Bucle de Entrenamiento con Early Stopping
import copy

best_val_loss     = float('inf')
epochs_no_improve = 0
best_model_state  = None

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_running_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        validar_batch_vs_grafo(x_batch, y_batch, edge_index, edge_weight, tag="train")

        optimizer.zero_grad()
        y_pred = model(x_batch, edge_index, edge_weight)
        loss   = criterion(y_pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        train_running_loss += loss.item()

    train_loss = train_running_loss / max(len(train_loader), 1)

    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            validar_batch_vs_grafo(x_batch, y_batch, edge_index, edge_weight, tag="val")
            val_running_loss += criterion(model(x_batch, edge_index, edge_weight), y_batch).item()

    val_loss = val_running_loss / max(len(val_loader), 1)
    curr_lr  = optimizer.param_groups[0]['lr']

    wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "lr": curr_lr})

    prev_lr = curr_lr
    scheduler.step(val_loss)
    curr_lr = optimizer.param_groups[0]['lr']
    lr_tag  = f"  ↓ lr={curr_lr:.2e}" if curr_lr < prev_lr else ""

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train: {train_loss:.6f} | Val: {val_loss:.6f}{lr_tag}"
          + (f"  [best]" if epochs_no_improve == 0 else ""))

    if epochs_no_improve >= ES_PATIENCE:
        print(f"\nEarly stopping en época {epoch} (sin mejora en {ES_PATIENCE} épocas).")
        break

# Restaurar el mejor modelo
model.load_state_dict(best_model_state)
print(f"\nMejor val loss: {best_val_loss:.6f}")

In [ ]:
# ─── Función reutilizable de entrenamiento ────────────────────────────────────
import time
import copy
import random as _rnd

# Grupos semánticos de features (índices sobre la dimensión F=14 de X_*_scaled)
GRUPOS_FEATURES = {
    'base_delay':  [0, 1, 2, 6, 7],       # delay_seconds, lagged_delay_1, lagged_delay_2,
                                           # route_rolling_delay, actual_headway_seconds
    'contexto':    [4, 5, 11, 12, 13],    # temp_extreme, n_eventos_afectando,
                                           # afecta_previo, afecta_durante, afecta_despues
    'calendario':  [8, 9, 10],             # hour_sin, hour_cos, dow
    'operativa':   [3],                    # is_unscheduled
}
ALL_IDX = list(range(14))

SUBSETS_ABLACION = {
    'all_features':    ALL_IDX,
    'sin_contexto':    [i for i in ALL_IDX if i not in GRUPOS_FEATURES['contexto']],
    'sin_calendario':  [i for i in ALL_IDX if i not in GRUPOS_FEATURES['calendario']],
    'sin_operativa':   [i for i in ALL_IDX if i not in GRUPOS_FEATURES['operativa']],
    'solo_base_delay': GRUPOS_FEATURES['base_delay'],
}


def train_one_config(params, feature_idx, max_epochs_override=None, seed=42, epoch_callback=None):
    """
    Entrena SubwayDCRNN con la configuración e índices de features indicados.

    Parámetros
    ----------
    params : dict
        Hiperparámetros: hidden_channels, K, dropout, lr, batch_size,
        history_len, grad_clip, scheduler_patience, scheduler_factor.
    feature_idx : list[int]
        Índices a seleccionar en la dimensión F de X_*_scaled.
    max_epochs_override : int | None
        Si se indica, sobreescribe NUM_EPOCHS.
    seed : int
        Semilla para numpy, torch y random.
    epoch_callback : callable | None
        f(epoch, val_loss) -> bool. True detiene el entrenamiento (pruner Optuna).

    Devuelve
    --------
    dict: best_val_loss_scaled, best_epoch, train_time_sec,
          best_model_state (None en trials de búsqueda para ahorrar ~2 GB),
          params, n_features, feature_idx, pruned
    """
    _rnd.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    hc  = params.get('hidden_channels',    HIDDEN_CHANNELS)
    K_v = params.get('K',                  K)
    drp = params.get('dropout',            DROPOUT)
    lr_ = params.get('lr',                 LR)
    bs  = params.get('batch_size',         batch_size)
    hl  = params.get('history_len',        history_len)
    gc_ = params.get('grad_clip',          GRAD_CLIP)
    sp  = params.get('scheduler_patience', 3)
    sf  = params.get('scheduler_factor',   0.5)
    n_ep = max_epochs_override if max_epochs_override is not None else NUM_EPOCHS
    nf   = len(feature_idx)

    X_tr = X_train_scaled[:, :, feature_idx]
    X_vl = X_val_scaled[:,   :, feature_idx]

    n_graph = int(edge_index.max().item()) + 1
    if X_tr.shape[1] != n_graph or Y_train_scaled.shape[1] != n_graph:
        raise RuntimeError(
            f"Desalineación grafo/datos en train_one_config: "
            f"X_tr.shape[1]={X_tr.shape[1]}, Y_train_scaled.shape[1]={Y_train_scaled.shape[1]}, n_graph={n_graph}. "
            f"Vuelve a ejecutar las celdas de construcción del grafo y escalado."
        )

    tr_ld = DataLoader(SubwayDataset(X_tr, Y_train_scaled, history_len=hl), batch_size=bs, shuffle=True)
    vl_ld = DataLoader(SubwayDataset(X_vl, Y_val_scaled,   history_len=hl), batch_size=bs, shuffle=False)

    mdl = SubwayDCRNN(
        in_channels=nf,
        hidden_channels=hc,
        out_horizons=OUT_HORIZONS,
        K=K_v,
        dropout=drp,
    ).to(device)
    opt  = torch.optim.Adam(mdl.parameters(), lr=lr_)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', patience=sp, factor=sf, min_lr=1e-5
    )
    crit = torch.nn.L1Loss()

    best_val   = float('inf')
    best_ep    = 0
    no_imp     = 0
    # No guardamos el estado del modelo en trials de búsqueda → ahorra ~2 GB por trial.
    # El entrenamiento final (celda de reentrenamiento) sí lo guarda.
    pruned     = False
    t0         = time.time()

    for ep in range(1, n_ep + 1):
        mdl.train()
        for xb, yb in tr_ld:
            xb, yb = xb.to(device), yb.to(device)
            validar_batch_vs_grafo(xb, yb, edge_index, edge_weight, tag="train_one_config-train")
            opt.zero_grad()
            loss = crit(mdl(xb, edge_index, edge_weight), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), gc_)
            opt.step()

        mdl.eval()
        with torch.no_grad():
            vl_acc = 0.0
            for xb, yb in vl_ld:
                xb, yb = xb.to(device), yb.to(device)
                validar_batch_vs_grafo(xb, yb, edge_index, edge_weight, tag="train_one_config-val")
                vl_acc += crit(mdl(xb, edge_index, edge_weight), yb).item()
            vl = vl_acc / len(vl_ld)
        sch.step(vl)

        if epoch_callback is not None and epoch_callback(ep, vl):
            pruned = True
            break

        if vl < best_val:
            best_val, best_ep, no_imp = vl, ep, 0
        else:
            no_imp += 1
        if no_imp >= ES_PATIENCE:
            break

    # Liberar el modelo del trial inmediatamente
    del mdl, opt, sch, tr_ld, vl_ld, X_tr, X_vl

    return {
        'best_val_loss_scaled': best_val,
        'best_epoch':           best_ep,
        'train_time_sec':       time.time() - t0,
        'best_model_state':     None,   # No se guarda en trials de búsqueda → ahorra ~2 GB
        'params':               params.copy(),
        'n_features':           nf,
        'feature_idx':          list(feature_idx),
        'pruned':               pruned,
    }

print("train_one_config definida correctamente.")
print(f"Subsets de ablacion disponibles: {list(SUBSETS_ABLACION.keys())}")


## Selección de variables por grupos (ablación de bajo coste)

Se evalúan 5 subconjuntos de features con un presupuesto reducido (`max_epochs=15`,
early stopping activo). El subconjunto con menor `val_loss` se usa en todas las fases
posteriores (tuning e inferencia final).


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  FASE 1 — Ablación de features (5 trials, epochs=15)        ║
# ╚══════════════════════════════════════════════════════════════╝
print("=" * 60)
print("FASE 1: Ablación de subconjuntos de features")
print("=" * 60)

PARAMS_BASE = {
    'hidden_channels':    HIDDEN_CHANNELS,
    'K':                  K,
    'dropout':            DROPOUT,
    'lr':                 LR,
    'batch_size':         batch_size,
    'history_len':        history_len,
    'grad_clip':          GRAD_CLIP,
    'scheduler_patience': 3,
    'scheduler_factor':   0.5,
}
MAX_EPOCHS_ABLACION = 15
N_ABLACION = len(SUBSETS_ABLACION)

ablacion_resultados = []
for trial_idx, (nombre_subset, idx) in enumerate(SUBSETS_ABLACION.items(), start=1):
    print(f"  Trial {trial_idx}/{N_ABLACION} | [{nombre_subset}] {len(idx)} features ...", end=' ', flush=True)
    res = train_one_config(PARAMS_BASE, idx, max_epochs_override=MAX_EPOCHS_ABLACION, seed=0)
    ablacion_resultados.append({
        'subset_name':    nombre_subset,
        'n_features':     res['n_features'],
        'val_loss':       res['best_val_loss_scaled'],
        'train_time_sec': res['train_time_sec'],
    })
    print(f"val_loss={res['best_val_loss_scaled']:.6f}  ({res['train_time_sec']:.0f}s)")
    del res

print("=" * 60)

df_ablacion = pd.DataFrame(ablacion_resultados).sort_values('val_loss').reset_index(drop=True)
del ablacion_resultados
print("\n=== Resultados de ablacion (ordenados por val_loss) ===")
print(df_ablacion.to_string(index=False))

selected_subset_name = df_ablacion.iloc[0]['subset_name']
selected_feature_set = SUBSETS_ABLACION[selected_subset_name]
print(f"\nSubconjunto seleccionado: '{selected_subset_name}' ({len(selected_feature_set)} features)")
print(f"Indices: {selected_feature_set}")


## Búsqueda de hiperparámetros — Estrategia 1: Random Search

Se muestrean 10 configuraciones aleatorias del espacio de hiperparámetros y se
entrena cada una con el subconjunto de features seleccionado.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  FASE 2 — Random Search (10 trials)                         ║
# ╚══════════════════════════════════════════════════════════════╝
print("=" * 60)
print("FASE 2: Random Search de hiperparámetros")
print("=" * 60)

import random as _rnd_rs

N_TRIALS_RANDOM = 10

ESPACIO_HP = {
    'hidden_channels':    [32, 64, 96],
    'K':                  [2, 3],
    'dropout':            (0.0, 0.4),
    'lr':                 (1e-4, 3e-3),
    'batch_size':         [8, 16],
    'history_len':        [4, 8, 12],
    'grad_clip':          [1.0, 3.0, 5.0],
    'scheduler_patience': [2, 3, 4],
    'scheduler_factor':   [0.3, 0.5, 0.7],
}

def muestrear_params_aleatorios(seed=None):
    rng = _rnd_rs.Random(seed)
    lr_min, lr_max = ESPACIO_HP['lr']
    return {
        'hidden_channels':    rng.choice(ESPACIO_HP['hidden_channels']),
        'K':                  rng.choice(ESPACIO_HP['K']),
        'dropout':            rng.uniform(*ESPACIO_HP['dropout']),
        'lr':                 10 ** rng.uniform(np.log10(lr_min), np.log10(lr_max)),
        'batch_size':         rng.choice(ESPACIO_HP['batch_size']),
        'history_len':        rng.choice(ESPACIO_HP['history_len']),
        'grad_clip':          rng.choice(ESPACIO_HP['grad_clip']),
        'scheduler_patience': rng.choice(ESPACIO_HP['scheduler_patience']),
        'scheduler_factor':   rng.choice(ESPACIO_HP['scheduler_factor']),
    }

random_resultados    = []
t_random_inicio      = time.time()
best_random_val      = float('inf')
best_random_params   = {}
best_random_time_sec = 0.0

for trial_num in range(N_TRIALS_RANDOM):
    params_trial = muestrear_params_aleatorios(seed=trial_num * 13 + 7)
    res = train_one_config(params_trial, selected_feature_set, seed=trial_num)
    row = {'trial': trial_num + 1, 'val_loss': res['best_val_loss_scaled'],
           'tiempo_sec': res['train_time_sec'], **params_trial}
    random_resultados.append(row)
    if res['best_val_loss_scaled'] < best_random_val:
        best_random_val      = res['best_val_loss_scaled']
        best_random_params   = params_trial.copy()
        best_random_time_sec = res['train_time_sec']
    print(f"  Trial {trial_num+1:02d}/{N_TRIALS_RANDOM} | val={res['best_val_loss_scaled']:.6f} | {res['train_time_sec']:.0f}s")
    del res, params_trial

t_random_total = time.time() - t_random_inicio

print("=" * 60)

df_random = pd.DataFrame(random_resultados).sort_values('val_loss').reset_index(drop=True)
del random_resultados
cols_vis = ['trial', 'val_loss', 'tiempo_sec', 'hidden_channels', 'K', 'lr', 'dropout']
print("\n=== Top-5 Random Search ===")
print(df_random[cols_vis].head(5).to_string(index=False))
print(f"\nMejor val_loss Random Search : {best_random_val:.6f}")
print(f"Tiempo total de busqueda      : {t_random_total:.0f}s")


In [ ]:
df_random.head(10)

## Búsqueda de hiperparámetros — Estrategia 2: Optuna (TPE + Pruning)

Se usa el algoritmo **TPE** (*Tree-structured Parzen Estimator*) que modela la distribución
de hiperparámetros buenos vs malos y focaliza los trials en regiones prometedoras. El
`MedianPruner` aborta trials que a mitad del entrenamiento están por encima de la mediana
de trials anteriores, reduciendo el coste total.


In [ ]:
# ─── Verificar disponibilidad de Optuna ──────────────────────────────────────
try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner
    OPTUNA_DISPONIBLE = True
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print(f"Optuna {optuna.__version__} disponible.")
except ImportError:
    OPTUNA_DISPONIBLE = False
    print("Optuna no instalado.")
    print("  Para instalar: uv add optuna  (o)  pip install optuna")
    print("  Fallback activo: se usaran los resultados de Random Search.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  FASE 3 — Optuna TPE + MedianPruner (8 trials)             ║
# ╚══════════════════════════════════════════════════════════════╝
print("=" * 60)
print("FASE 3: Optuna con TPE + MedianPruner")
print("=" * 60)

N_TRIALS_OPTUNA      = 8
optuna_resultados    = []
t_optuna_total       = 0.0
best_optuna_val      = float('inf')
best_optuna_params   = {}
best_optuna_time_sec = 0.0

if OPTUNA_DISPONIBLE:
    def objetivo_optuna(trial):
        params_trial = {
            'hidden_channels':    trial.suggest_categorical('hidden_channels', [32, 64, 96]),
            'K':                  trial.suggest_categorical('K', [2, 3]),
            'dropout':            trial.suggest_float('dropout', 0.0, 0.4),
            'lr':                 trial.suggest_float('lr', 1e-4, 3e-3, log=True),
            'batch_size':         trial.suggest_categorical('batch_size', [8, 16]),
            'history_len':        trial.suggest_categorical('history_len', [4, 8, 12]),
            'grad_clip':          trial.suggest_categorical('grad_clip', [1.0, 3.0, 5.0]),
            'scheduler_patience': trial.suggest_categorical('scheduler_patience', [2, 3, 4]),
            'scheduler_factor':   trial.suggest_categorical('scheduler_factor', [0.3, 0.5, 0.7]),
        }

        def callback_pruner(ep, vl):
            trial.report(vl, ep)
            return trial.should_prune()

        res = train_one_config(
            params_trial, selected_feature_set,
            seed=trial.number, epoch_callback=callback_pruner
        )
        val_loss = res['best_val_loss_scaled']
        pruned   = res['pruned']

        del res, params_trial
        print(f"  Trial {trial.number+1:02d}/{N_TRIALS_OPTUNA} | val={val_loss:.6f}")

        if pruned:
            raise optuna.exceptions.TrialPruned()
        return val_loss

    study = optuna.create_study(
        direction='minimize',
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=2),  # n_warmup_steps reducido a 2
    )
    t_optuna_inicio = time.time()
    study.optimize(objetivo_optuna, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
    t_optuna_total = time.time() - t_optuna_inicio

    for t in study.trials:
        dur = t.duration.total_seconds() if t.duration else 0.0
        if t.state == optuna.trial.TrialState.COMPLETE:
            optuna_resultados.append(
                {'trial': t.number + 1, 'val_loss': t.value, 'tiempo_sec': dur, **t.params}
            )
        elif t.state == optuna.trial.TrialState.PRUNED:
            optuna_resultados.append(
                {'trial': t.number + 1, 'val_loss': float('nan'), 'tiempo_sec': dur, **t.params}
            )

    df_optuna = pd.DataFrame(optuna_resultados)
    del optuna_resultados
    df_optuna_ok = df_optuna.dropna(subset=['val_loss']).sort_values('val_loss').reset_index(drop=True)

    best_optuna_val      = study.best_value
    best_optuna_params   = dict(study.best_params)
    best_optuna_time_sec = (
        study.best_trial.duration.total_seconds() if study.best_trial.duration else 0.0
    )
    n_podados = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)

    cols_vis = ['trial', 'val_loss', 'tiempo_sec', 'hidden_channels', 'K', 'lr', 'dropout']
    cols_disp = [c for c in cols_vis if c in df_optuna_ok.columns]
    print("=== Top-5 Optuna ===")
    print(df_optuna_ok[cols_disp].head(5).to_string(index=False))
    print(f"\nMejor val_loss Optuna   : {best_optuna_val:.6f}")
    print(f"Trials podados          : {n_podados}/{N_TRIALS_OPTUNA}")
    print(f"Tiempo total busqueda   : {t_optuna_total:.0f}s")

else:
    print("Optuna no disponible — usando resultados de Random Search como fallback.")
    df_optuna     = df_random.copy()
    df_optuna_ok  = df_random.copy()
    best_optuna_val      = best_random_val
    best_optuna_params   = best_random_params.copy()
    best_optuna_time_sec = best_random_time_sec
    t_optuna_total       = t_random_total
    N_TRIALS_OPTUNA      = N_TRIALS_RANDOM

print("=" * 60)


## Comparación de estrategias: Random Search vs Optuna


In [ ]:
# ─── Tabla comparativa de estrategias ────────────────────────────────────────
strategy_comparison = pd.DataFrame([
    {
        'strategy':              'random',
        'n_trials':              N_TRIALS_RANDOM,
        'best_val_loss':         best_random_val,
        'total_search_time_sec': t_random_total,
        'best_trial_time_sec':   best_random_time_sec,
        'val_loss_per_min':      best_random_val / (t_random_total / 60),
    },
    {
        'strategy':              'optuna',
        'n_trials':              N_TRIALS_OPTUNA,
        'best_val_loss':         best_optuna_val,
        'total_search_time_sec': t_optuna_total,
        'best_trial_time_sec':   best_optuna_time_sec,
        'val_loss_per_min':      best_optuna_val / (t_optuna_total / 60),
    },
])

print("=== Comparacion de estrategias de busqueda ===")
print(strategy_comparison.to_string(index=False))

mejor_calidad  = strategy_comparison.loc[strategy_comparison['best_val_loss'].idxmin(),              'strategy']
mejor_tiempo   = strategy_comparison.loc[strategy_comparison['total_search_time_sec'].idxmin(), 'strategy']
diff_pct = (abs(best_random_val - best_optuna_val)
            / max(best_random_val, best_optuna_val) * 100)
t_diff   = abs(t_random_total - t_optuna_total)

print(f"\nMejor en calidad (menor val_loss)       : {mejor_calidad.upper()}")
print(f"Mejor en eficiencia (menor tiempo total) : {mejor_tiempo.upper()}")
print(f"Diferencia de calidad                    : {diff_pct:.2f}%")
print(f"Diferencia de tiempo                     : {t_diff:.0f}s")
if diff_pct < 2.0:
    print(f"\nCompromiso: diferencia de calidad < 2% -> ambas estrategias son equivalentes "
          f"para este presupuesto ({N_TRIALS_RANDOM} random / {N_TRIALS_OPTUNA} optuna trials).")
elif mejor_calidad == mejor_tiempo:
    print(f"\nCompromiso: {mejor_calidad.upper()} domina en calidad Y tiempo -> eleccion clara.")
else:
    print(f"\nCompromiso: {mejor_calidad.upper()} es mejor en calidad pero "
          f"{mejor_tiempo.upper()} es mas rapido. Depende del presupuesto disponible.")


## Entrenamiento final con la mejor configuración

Se selecciona la estrategia con menor `val_loss` y se reentrena el modelo con:
- El subconjunto de features identificado en la fase de ablación.
- Los mejores hiperparámetros encontrados por la estrategia ganadora.
- El número completo de épocas (`NUM_EPOCHS`) con early stopping.

**El conjunto de test se evalúa una única vez al final**, sin intervención en ninguna
decisión de entrenamiento o selección de hiperparámetros.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  FASE 4 — Entrenamiento final (epochs completas)            ║
# ╚══════════════════════════════════════════════════════════════╝
print("=" * 60)
print("Entrenamiento final con la mejor configuración")
print("=" * 60)

if best_random_val <= best_optuna_val:
    best_strategy     = 'random'
    best_final_params = best_random_params.copy()
    best_final_val    = best_random_val
else:
    best_strategy     = 'optuna'
    best_final_params = best_optuna_params.copy()
    best_final_val    = best_optuna_val

print(f"Estrategia seleccionada  : {best_strategy.upper()}")
print(f"val_loss de referencia   : {best_final_val:.6f}")
print(f"Subconjunto de features  : '{selected_subset_name}' ({len(selected_feature_set)} features)")
print("Hiperparametros:")
for k, v in best_final_params.items():
    print(f"  {k}: {v}")

print("\nEntrenando modelo final (epochs completas)...")

# El entrenamiento final necesita guardar el estado → reentrenamos con copy.deepcopy activo.
# Usamos directamente el bucle completo (no train_one_config) para poder guardar best_state.
import copy as _copy

_rnd.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

hc_f  = best_final_params.get('hidden_channels',    HIDDEN_CHANNELS)
K_f   = best_final_params.get('K',                  K)
drp_f = best_final_params.get('dropout',            DROPOUT)
lr_f  = best_final_params.get('lr',                 LR)
bs_f  = best_final_params.get('batch_size',         batch_size)
hl_f  = best_final_params.get('history_len',        history_len)
gc_f  = best_final_params.get('grad_clip',          GRAD_CLIP)
sp_f  = best_final_params.get('scheduler_patience', 3)
sf_f  = best_final_params.get('scheduler_factor',   0.5)
nf_f  = len(selected_feature_set)

X_tr_f = X_train_scaled[:, :, selected_feature_set]
X_vl_f = X_val_scaled[:,   :, selected_feature_set]

tr_ld_f = DataLoader(SubwayDataset(X_tr_f, Y_train_scaled, history_len=hl_f), batch_size=bs_f, shuffle=True)
vl_ld_f = DataLoader(SubwayDataset(X_vl_f, Y_val_scaled,   history_len=hl_f), batch_size=bs_f, shuffle=False)

model_final = SubwayDCRNN(
    in_channels=nf_f,
    hidden_channels=hc_f,
    out_horizons=OUT_HORIZONS,
    K=K_f,
    dropout=drp_f,
).to(device)

opt_f  = torch.optim.Adam(model_final.parameters(), lr=lr_f)
sch_f  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_f, mode='min', patience=sp_f, factor=sf_f, min_lr=1e-5)
crit_f = torch.nn.L1Loss()

best_val_f   = float('inf')
best_ep_f    = 0
no_imp_f     = 0
best_state_f = None
t0_f         = time.time()

for ep in range(1, NUM_EPOCHS + 1):
    model_final.train()
    for xb, yb in tr_ld_f:
        xb, yb = xb.to(device), yb.to(device)
        opt_f.zero_grad()
        loss = crit_f(model_final(xb, edge_index, edge_weight), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_final.parameters(), gc_f)
        opt_f.step()

    model_final.eval()
    with torch.no_grad():
        vl_acc = sum(
            crit_f(model_final(xb.to(device), edge_index, edge_weight), yb.to(device)).item()
            for xb, yb in vl_ld_f
        )
        vl_f = vl_acc / len(vl_ld_f)
    sch_f.step(vl_f)

    if vl_f < best_val_f:
        best_val_f, best_ep_f, no_imp_f = vl_f, ep, 0
        best_state_f = _copy.deepcopy(model_final.state_dict())
    else:
        no_imp_f += 1
    print(f"  Época {ep:02d}/{NUM_EPOCHS} | val={vl_f:.6f}"
          + ("  [best]" if no_imp_f == 0 else ""))
    if no_imp_f >= ES_PATIENCE:
        print(f"  Early stopping en época {ep}.")
        break

tiempo_final = time.time() - t0_f
del tr_ld_f, vl_ld_f, X_tr_f, X_vl_f, opt_f, sch_f

print(f"\nEntrenamiento final completado.")
print(f"  Mejor val_loss : {best_val_f:.6f}  (época {best_ep_f})")
print(f"  Tiempo         : {tiempo_final:.0f}s")

# Cargar el mejor estado — validar antes de llamar a load_state_dict
if best_state_f is not None:
    model_final.load_state_dict(best_state_f)
    print("Mejor estado del modelo cargado correctamente.")
else:
    print("AVISO: best_model_state es None; se usan los pesos del último epoch.")

# Exponer resultado compatible con la celda de evaluación
hl_final = hl_f
res_final = {
    'best_val_loss_scaled': best_val_f,
    'best_epoch':           best_ep_f,
    'train_time_sec':       tiempo_final,
    'best_model_state':     best_state_f,
}

print("=" * 60)


In [ ]:
# ─── Evaluacion en test + registro W&B + tabla resumen ───────────────────────
X_te_sel    = X_test_scaled[:, :, selected_feature_set]
bs_fin      = best_final_params.get('batch_size', batch_size)
test_ds_fin = SubwayDataset(X_te_sel, Y_test_scaled, history_len=hl_final)
test_ld_fin = DataLoader(test_ds_fin, batch_size=bs_fin, shuffle=False)

model_final.eval()
preds_fin, trues_fin = [], []
with torch.no_grad():
    for xb, yb in test_ld_fin:
        xb = xb.to(device)
        validar_batch_vs_grafo(xb, yb, edge_index, edge_weight, tag="test")
        preds_fin.append(model_final(xb, edge_index, edge_weight).cpu().numpy())
        trues_fin.append(yb.numpy())

preds_fin = np.concatenate(preds_fin, axis=0).squeeze(1)   # (T, N, 3)
trues_fin = np.concatenate(trues_fin, axis=0).squeeze(1)   # (T, N, 3)

T_f, N_f, H_f = preds_fin.shape
preds_inv_fin = scaler_Y.inverse_transform(preds_fin.reshape(-1, H_f)).reshape(T_f, N_f, H_f)
trues_inv_fin = scaler_Y.inverse_transform(trues_fin.reshape(-1, H_f)).reshape(T_f, N_f, H_f)

horizontes = ['10m', '20m', '30m']
mae_fin = {}
print("=== MAE final en test (segundos reales) ===")
for i, h in enumerate(horizontes):
    mae = float(np.abs(preds_inv_fin[:, :, i] - trues_inv_fin[:, :, i]).mean())
    mae_fin[f'test_mae_{h}'] = mae
    print(f"  station_delay_{h}: {mae:.1f} s")
mae_fin['test_mae_mean'] = float(np.mean(list(mae_fin.values())))
print(f"  Media            : {mae_fin['test_mae_mean']:.1f} s")

# ── Nuevo run W&B para el experimento de tuning ──
try:
    wandb.init(
        project=WANDB_PROJECT,
        name="dcrnn-tuned-final",
        config={
            'best_strategy':   best_strategy,
            'selected_subset': selected_subset_name,
            'n_features':      len(selected_feature_set),
            'feature_idx':     selected_feature_set,
            **{f'hp_{k}': v for k, v in best_final_params.items()},
            'val_loss_final':  res_final['best_val_loss_scaled'],
        }
    )
    wandb.log({**mae_fin, 'val_loss_final': res_final['best_val_loss_scaled']})
    wandb.log({
        'ablacion':            wandb.Table(dataframe=df_ablacion),
        'strategy_comparison': wandb.Table(dataframe=strategy_comparison),
    })
    wandb.finish()
    print("\nResultados registrados en W&B (run: dcrnn-tuned-final).")
except Exception as e:
    print(f"\nW&B no disponible o error ({e}). Resultados solo impresos en notebook.")

# ── Tabla resumen final ──
print("\n=== RESUMEN FINAL ===")
resumen_final = pd.DataFrame([{
    'best_feature_subset': selected_subset_name,
    'n_features':          len(selected_feature_set),
    'best_strategy':       best_strategy,
    'val_loss_final':      round(res_final['best_val_loss_scaled'], 6),
    'test_mae_10m':        round(mae_fin['test_mae_10m'], 1),
    'test_mae_20m':        round(mae_fin['test_mae_20m'], 1),
    'test_mae_30m':        round(mae_fin['test_mae_30m'], 1),
    'test_mae_mean':       round(mae_fin['test_mae_mean'], 1),
}])
print(resumen_final.T.to_string(header=False))

## Análisis comparado: Random Search vs Optuna

- **Random Search** explora el espacio de forma uniforme y sin memoria entre trials.
  Su rendimiento depende del presupuesto de trials y la aleatoriedad del muestreo.
  Con 10 trials puede no ser suficiente para explorar el espacio de hiperparámetros.
- **Optuna (TPE)** construye un modelo probabilístico del espacio de búsqueda y
  focaliza trials en regiones prometedoras. Generalmente obtiene mejor `val_loss`
  con el mismo presupuesto, especialmente en hiperparámetros continuos como `lr`.

### Coste y eficiencia
- Random Search tiene coste de infraestructura nulo y es trivialmente paralelizable.
- Optuna introduce un overhead de modelado TPE (despreciable en pocos trials) pero
  reduce el tiempo total mediante el `MedianPruner`: los trials claramente malos se
  abortan pronto, ahorrando ciclos de GPU/CPU.

### Conclusión
En este contexto (espacio de dimensionalidad reducida, 8-10 trials), Random Search ofrece un mejor resultado,
ya que el número de trials no es suficiente para que se aproveche Optuna al máximo. Para trabajos futuros se
recomienda aumentar el presupuesto a 20–50 trials e incluso activar paralelismo con
`study.optimize(..., n_jobs=-1)`.
